<div style="border-left:6px solid #ae0000; padding:6px 20px; margin-bottom:4px;">
<h1 style="margin:0; color:#0d2741;">Fundamentos de Inferencia Estadística</h1>
<p style="margin:4px 0 0 0; color:#0d2741; font-size:1.15em;">De la Muestra a la Población</p>
<p style="margin:6px 0 0 0; color:#444; font-size:1.05em;"><em>Estadística Computacional para la Toma de Decisiones</em></p>
</div>

**Magíster en Ciencia de Datos e Inteligencia Artificial** · Universidad Andrés Bello  
Maidana, J.P. (2026)

---

> **Cómo usar este notebook.** Ejecuta las celdas en orden. Comienza por **«Preparación del entorno»**. Las ecuaciones se muestran con notación matemática (LaTeX) y los conceptos clave se acompañan de **simulaciones ejecutables** que validan la teoría empíricamente.

## Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.patches import Ellipse, FancyBboxPatch   # solo para los diagramas

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
plt.rcParams['figure.dpi'] = 100

print("numpy:", np.__version__, "| scipy:", __import__('scipy').__version__)

## 1. Introducción

> *"No podemos medir el océano completo, pero podemos tomar muestras de agua para conocer su composición."*

La **inferencia estadística** es el proceso fundamental que permite usar datos de una muestra para hacer generalizaciones sobre una población más grande. Es el puente entre lo observado (nuestra muestra) y lo desconocido (la población completa).

### 1.1 ¿Por qué es Necesaria la Inferencia?

En la práctica, rara vez tenemos acceso a poblaciones completas:

- **Control de calidad:** inspeccionar cada producto destruiría el inventario.
- **Estudios médicos:** imposible probar medicamentos en toda la población.
- **Encuestas políticas:** prohibitivamente costoso encuestar a millones.
- **A/B Testing:** probar con muestra antes de implementar a escala.
- **Análisis de datos:** datasets completos pueden ser inmanejables.

<div style="background-color:#eef2f7; border:1px solid #b0bec5; border-radius:6px; padding:12px 18px; margin:8px 0; text-align:center;">
<strong>Pregunta Central de la Inferencia</strong><br><br>
<em>¿Cómo podemos usar información de una muestra para hacer afirmaciones confiables sobre toda la población, cuantificando la incertidumbre inherente a este proceso?</em>
</div>

### 1.2 El Ecosistema de la Inferencia

El siguiente diagrama resume la relación entre población, muestra e inferencia.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 13); ax.set_ylim(-3.5, 3.5); ax.axis('off')

# Población
ax.add_patch(Ellipse((2.6, 0), 5.0, 5.0, fc='#d6e4f0', ec='#1565c0', lw=2))
ax.text(2.6, 1.7, "POBLACIÓN", ha='center', fontsize=14, fontweight='bold', color='#1565c0')
ax.text(2.6, 1.0, "$N$ elementos", ha='center', fontsize=10)
ax.text(2.6, 0.3, "Parámetros (desconocidos)", ha='center', fontsize=11, fontweight='bold')
ax.text(2.6, -0.7, r"$\mu$ = media" + "\n" + r"$\sigma^2$ = varianza" + "\n" + r"$p$ = proporción",
        ha='center', fontsize=9, va='top')

# Muestra
ax.add_patch(Ellipse((10.4, 0), 3.4, 3.0, fc='#dcedc8', ec='#2e7d32', lw=2))
ax.text(10.4, 1.0, "MUESTRA", ha='center', fontsize=13, fontweight='bold', color='#2e7d32')
ax.text(10.4, 0.45, "$n$ elementos", ha='center', fontsize=9)
ax.text(10.4, -0.05, "Estadísticos (observados)", ha='center', fontsize=10, fontweight='bold')
ax.text(10.4, -0.6, r"$\bar{x}$,  $s^2$,  $\hat{p}$", ha='center', fontsize=11, va='top')

# Flecha muestreo
ax.annotate('', xy=(8.5, 0.5), xytext=(5.2, 0.5),
    arrowprops=dict(arrowstyle='-|>', color='#c62828', lw=2.5))
ax.text(6.85, 0.9, "MUESTREO", ha='center', fontsize=10, fontweight='bold', color='#c62828')
ax.text(6.85, 0.15, "aleatorio", ha='center', fontsize=8, color='#c62828')

# Flecha inferencia
ax.annotate('', xy=(5.2, -0.6), xytext=(8.5, -0.6),
    arrowprops=dict(arrowstyle='-|>', color='#e65100', lw=2.5, linestyle='--'))
ax.text(6.85, -1.05, "INFERENCIA", ha='center', fontsize=10, fontweight='bold', color='#e65100')
ax.text(6.85, -0.35, "generalización", ha='center', fontsize=8, color='#e65100')

# Caja incertidumbre
ax.add_patch(FancyBboxPatch((5.35, -2.85), 3.0, 0.8,
    boxstyle="round,pad=0.05", fc='#f3e5f5', ec='#7b1fa2', lw=1.5))
ax.text(6.85, -2.45, "Cuantificar incertidumbre", ha='center', fontsize=9, fontweight='bold', color='#4a148c')
ax.annotate('', xy=(6.85, -2.05), xytext=(6.85, -1.4),
    arrowprops=dict(arrowstyle='-|>', color='#7b1fa2', lw=1.5))

plt.tight_layout()
plt.show()

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Objetivos de Aprendizaje</p>
<ol style="margin:0;">
<li>Distinguir claramente entre población y muestra, parámetros y estadísticos.</li>
<li>Comprender el concepto de distribución muestral.</li>
<li>Dominar el Teorema del Límite Central (TLC) y sus aplicaciones.</li>
<li>Calcular e interpretar el error estándar.</li>
<li>Construir intervalos de confianza.</li>
<li>Implementar simulaciones de inferencia en Python.</li>
<li>Aplicar inferencia a problemas reales de toma de decisiones.</li>
</ol>
</div>

## 2. Marco Conceptual: Población vs Muestra

### 2.1 Definiciones Fundamentales

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Población ($N$)</p>
<p style="margin:0 0 6px 0;">Conjunto completo de todos los elementos de interés en un estudio.</p>
<p style="margin:0 0 4px 0;"><strong>Características:</strong></p>
<ul style="margin:0 0 8px 0;">
<li>Contiene TODOS los elementos.</li>
<li>Tiene parámetros fijos (pero generalmente desconocidos).</li>
<li>Puede ser finita o infinita.</li>
<li>Usualmente imposible o impráctico de estudiar completamente.</li>
</ul>
<p style="margin:0 0 4px 0;"><strong>Parámetros poblacionales</strong> (letras griegas): $\mu$ (media), $\sigma^2$ (varianza), $\sigma$ (desviación estándar), $p$ (proporción).</p>
</div>

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Muestra ($n$)</p>
<p style="margin:0 0 6px 0;">Subconjunto de la población seleccionado para estudio.</p>
<p style="margin:0 0 4px 0;"><strong>Características:</strong></p>
<ul style="margin:0 0 8px 0;">
<li>Contiene solo PARTE de la población.</li>
<li>Genera estadísticos observables.</li>
<li>Debe ser representativa (idealmente aleatoria).</li>
<li>Práctica y económica de estudiar.</li>
</ul>
<p style="margin:0 0 4px 0;"><strong>Estadísticos muestrales</strong> (letras latinas): $\bar{x}$ (media), $s^2$ (varianza), $s$ (desviación estándar), $\hat{p}$ (proporción).</p>
</div>

### 2.2 Tabla de Notación Estándar

| Concepto | Población | Muestra | Relación | Nombre Muestral |
|---|:---:|:---:|:---:|---|
| Tamaño | $N$ | $n$ | $n \ll N$ | Tamaño muestral |
| Media | $\mu$ | $\bar{x}$ | $\bar{x} \approx \mu$ | Media muestral |
| Varianza | $\sigma^2$ | $s^2$ | $s^2 \approx \sigma^2$ | Varianza muestral |
| Desv. Est. | $\sigma$ | $s$ | $s \approx \sigma$ | Desviación est. muestral |
| Proporción | $p$ | $\hat{p}$ | $\hat{p} \approx p$ | Proporción muestral |

<div align="center"><em>Tabla 2.1 — Notación estándar: parámetros poblacionales vs estadísticos muestrales.</em></div>

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Principio Fundamental de la Inferencia</p>
<p style="margin:0 0 6px 0;">Los estadísticos muestrales ($\bar{x}$, $s^2$, $\hat{p}$) son <strong>variables aleatorias</strong>:</p>
<ul style="margin:0 0 8px 0;">
<li>Varían de muestra en muestra.</li>
<li>Su variación es predecible y cuantificable.</li>
<li>Siguen distribuciones de probabilidad conocidas (distribuciones muestrales).</li>
</ul>
<p style="margin:0 0 4px 0;">Esta predictibilidad permite:</p>
<ol style="margin:0;">
<li><strong>Estimación puntual:</strong> $\bar{x}$ estima $\mu$.</li>
<li><strong>Estimación por intervalo:</strong> intervalos de confianza.</li>
<li><strong>Pruebas de hipótesis:</strong> evaluar afirmaciones sobre parámetros.</li>
<li><strong>Cuantificación de incertidumbre:</strong> error estándar.</li>
</ol>
</div>

### 2.3 Muestreo Aleatorio

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Muestreo Aleatorio Simple (MAS)</p>
<p style="margin:0 0 6px 0;">Cada elemento de la población tiene la misma probabilidad de ser seleccionado, y las selecciones son independientes.</p>
<p style="margin:0 0 4px 0;"><strong>Por qué es crítico:</strong></p>
<ul style="margin:0 0 8px 0;">
<li><strong>Elimina sesgos:</strong> no favorece subgrupos.</li>
<li><strong>Representatividad:</strong> la muestra refleja la población.</li>
<li><strong>Base matemática:</strong> permite aplicar teoría de probabilidad.</li>
<li><strong>Generalización válida:</strong> resultados extrapolables.</li>
</ul>
<p style="margin:0;"><strong>Tipos:</strong> sin reemplazo (el elemento no vuelve a la población, más común) y con reemplazo (puede seleccionarse múltiples veces, base del bootstrap).</p>
</div>

<div style="background-color:#fdecea; border-left:5px solid #c62828; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#b71c1c; font-size:1.05em;">⚠️&nbsp; Advertencia — Muestreo NO Aleatorio — Fuentes de Sesgo</p>
<ul style="margin:0 0 8px 0;">
<li><strong>Sesgo de conveniencia:</strong> encuestar solo a quien está disponible.</li>
<li><strong>Sesgo de selección:</strong> solo responden los voluntarios.</li>
<li><strong>Sesgo de supervivencia:</strong> solo se observan casos exitosos.</li>
<li><strong>Sesgo de no respuesta:</strong> sistemáticamente algunos no responden.</li>
</ul>
<p style="margin:0;"><strong>Consecuencia:</strong> inferencias inválidas, conclusiones erróneas.</p>
</div>

## 3. Distribuciones Muestrales

### 3.1 Concepto Fundamental

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Distribución Muestral</p>
<p style="margin:0 0 6px 0;">La distribución de probabilidad de todos los valores posibles que un estadístico puede tomar en todas las muestras posibles de tamaño $n$ de una población.</p>
<p style="margin:0 0 4px 0;"><strong>No es:</strong></p>
<ul style="margin:0 0 8px 0;">
<li>La distribución de los datos en UNA muestra.</li>
<li>La distribución de la población.</li>
</ul>
<p style="margin:0 0 4px 0;"><strong>Sí es:</strong></p>
<ul style="margin:0;">
<li>La distribución del estadístico (ej. $\bar{x}$) a través de TODAS las muestras posibles.</li>
<li>La distribución de probabilidad de $\bar{x}$ si repitiéramos el muestreo infinitas veces.</li>
</ul>
</div>

> **Visualización conceptual.** Si de una población con $\mu = 50$ y $\sigma = 10$ tomáramos muchísimas muestras de tamaño $n$, cada una daría una media muestral $\bar{x}$ ligeramente distinta (48.2, 51.3, 49.7, …). El histograma de **todas esas medias** es la distribución muestral de $\bar{x}$. La simulación de la Sección 6.1 construye exactamente esa distribución de forma empírica.

### 3.2 Propiedades de la Distribución Muestral de $\bar{x}$

<div style="background-color:#f5ecfb; border-left:5px solid #7b1fa2; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#4a148c; font-size:1.05em;">📐&nbsp; Teorema — Propiedades de la media muestral</p>
<p style="margin:0 0 8px 0;">Si $X_1, X_2, \ldots, X_n$ es una muestra aleatoria de tamaño $n$ de una población con media $\mu$ y desviación estándar $\sigma$, entonces:</p>
<p style="margin:0 0 2px 0;"><strong>1. Esperanza (centro):</strong></p>
$$E[\bar{X}] = \mu$$
<p style="margin:0 0 8px 0;">La media de todas las medias muestrales es igual a la media poblacional: $\bar{X}$ es un <strong>estimador insesgado</strong> de $\mu$.</p>
<p style="margin:0 0 2px 0;"><strong>2. Varianza (dispersión):</strong></p>
$$\operatorname{Var}(\bar{X}) = \frac{\sigma^2}{n}$$
<p style="margin:0 0 8px 0;">La variabilidad de $\bar{X}$ disminuye con el tamaño muestral.</p>
<p style="margin:0 0 2px 0;"><strong>3. Error estándar (precisión):</strong></p>
$$SE(\bar{X}) = \sqrt{\operatorname{Var}(\bar{X})} = \frac{\sigma}{\sqrt{n}}$$
<p style="margin:0;">Desviación estándar de la distribución muestral; cuantifica la precisión del estimador.</p>
</div>

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Implicaciones Críticas</p>
<ol style="margin:0;">
<li><strong>Insesgamiento:</strong> en promedio, $\bar{x}$ acierta al verdadero $\mu$.</li>
<li><strong>Efecto de $n$:</strong> $SE \propto 1/\sqrt{n}$. Duplicar $n$ reduce el $SE$ en ≈ 29% (factor $1/\sqrt{2}$); cuadruplicar $n$ reduce el $SE$ a la mitad; la reducción marginal disminuye con $n$ grande.</li>
<li><strong>Trade-off costo-precisión:</strong> más datos = mayor precisión, pero también mayor costo y tiempo (ley de rendimientos decrecientes).</li>
</ol>
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Efecto del Tamaño Muestral</p>
<p style="margin:0 0 8px 0;">Población: $\mu = 100$, $\sigma = 20$.</p>
<table style="border-collapse:collapse; width:100%;">
<tr style="border-bottom:2px solid #888;"><th align="left">n</th><th align="left">SE</th><th align="left">Reducción</th><th align="left">Interpretación</th></tr>
<tr><td>25</td><td>$20/\sqrt{25} = 4.0$</td><td>—</td><td>Línea base</td></tr>
<tr><td>100</td><td>$20/\sqrt{100} = 2.0$</td><td>50%</td><td>4× muestra, mitad de SE</td></tr>
<tr><td>400</td><td>$20/\sqrt{400} = 1.0$</td><td>75%</td><td>16× muestra, 1/4 de SE</td></tr>
<tr><td>1600</td><td>$20/\sqrt{1600} = 0.5$</td><td>87.5%</td><td>64× muestra, 1/8 de SE</td></tr>
</table>
<p style="margin:8px 0 0 0;"><strong>Conclusión:</strong> para reducir el SE a la mitad, hay que cuadruplicar $n$.</p>
</div>

## 4. Teorema del Límite Central (TLC)

### 4.1 Enunciado Formal

<div style="background-color:#f5ecfb; border-left:5px solid #7b1fa2; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#4a148c; font-size:1.05em;">📐&nbsp; Teorema — Teorema del Límite Central (CLT)</p>
<p style="margin:0 0 8px 0;">Sea $X_1, X_2, \ldots, X_n$ una muestra aleatoria de tamaño $n$ de una población con media $\mu$ y varianza finita $\sigma^2$. Entonces, cuando $n$ es suficientemente grande:</p>
$$\bar{X} \;\sim\; N\!\left(\mu,\; \frac{\sigma^2}{n}\right) \quad \text{(aproximadamente)}$$
<p style="margin:8px 0 2px 0;">O equivalentemente, la variable estandarizada:</p>
$$Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}} = \frac{\bar{X} - \mu}{SE(\bar{X})} \;\sim\; N(0, 1)$$
<p style="margin:8px 0 4px 0;"><strong>Condiciones:</strong></p>
<ul style="margin:0;">
<li>Muestra aleatoria e independiente.</li>
<li>Tamaño muestral suficientemente grande (regla práctica: $n \geq 30$).</li>
<li>Varianza poblacional finita ($\sigma^2 < \infty$).</li>
</ul>
</div>

<div style="background-color:#eef2f7; border:1px solid #b0bec5; border-radius:6px; padding:12px 18px; margin:8px 0; text-align:center;">
<strong>Resultado Revolucionario</strong><br><br>
<em>No importa la forma de la distribución poblacional original (puede ser asimétrica, uniforme, bimodal, discreta, etc.):<br>
la distribución de $\bar{X}$ SIEMPRE converge a Normal.</em>
</div>

### 4.2 Ilustración Visual del TLC

La siguiente figura toma una población **exponencial** (fuertemente asimétrica) y muestra cómo la distribución muestral de $\bar{x}$ se vuelve cada vez más normal a medida que crece $n$. La curva roja es la Normal teórica y el valor de asimetría (*skew*) confirma la convergencia.

In [ ]:
# Población exponencial (muy asimétrica) y convergencia de la media muestral
np.random.seed(123)
pob = np.random.exponential(scale=10, size=100000)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

# Panel 0: población
axes[0].hist(pob, bins=60, density=True, color='#ef9a9a', edgecolor='white')
axes[0].set_title(f'Población Exponencial\n(skew = {stats.skew(pob):.2f})', fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('Densidad')

# Paneles 1-5: distribución muestral de x_barra para distintos n
for idx, n in enumerate([2, 5, 10, 30, 100], start=1):
    medias = np.array([np.random.choice(pob, size=n, replace=True).mean()
                       for _ in range(8000)])
    ax = axes[idx]
    ax.hist(medias, bins=40, density=True, color='#90caf9', edgecolor='white')
    x = np.linspace(medias.min(), medias.max(), 200)
    ax.plot(x, stats.norm.pdf(x, medias.mean(), medias.std()), 'r-', lw=2)
    ax.set_title(f'n = {n}   (skew = {stats.skew(medias):.2f})', fontweight='bold')
    ax.set_xlabel(r'$\bar{x}$')

fig.suptitle('Teorema del Límite Central: convergencia de $\\bar{x}$ a Normal',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — ¿Qué tan grande debe ser $n$?</p>
<p style="margin:0 0 4px 0;"><strong>Regla práctica:</strong></p>
<ul style="margin:0 0 8px 0;">
<li>$n \geq 30$: suficiente para la mayoría de distribuciones.</li>
<li>$n \geq 50$: recomendado si la población es muy asimétrica.</li>
<li>$n < 30$: el TLC puede no aplicar bien.</li>
</ul>
<p style="margin:0 0 4px 0;"><strong>Casos especiales:</strong></p>
<ul style="margin:0 0 8px 0;">
<li>Si la población YA es normal: el TLC aplica para cualquier $n$ (incluso $n=1$).</li>
<li>Si la población es simétrica: $n \geq 15$ suele bastar.</li>
<li>Si hay outliers extremos: puede requerir $n > 50$.</li>
</ul>
<p style="margin:0;"><strong>Cuando el TLC NO aplica bien</strong> ($n$ pequeño Y población muy asimétrica): usar la distribución $t$ de Student o métodos no paramétricos.</p>
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Industrias donde el TLC es Fundamental</p>
<p style="margin:0 0 6px 0;"><strong>1. Control de calidad:</strong> los promedios de mediciones siguen distribución normal, lo que permite establecer límites de control ($\mu \pm 3\sigma/\sqrt{n}$). Six Sigma se basa en la normalidad de los promedios.</p>
<p style="margin:0 0 6px 0;"><strong>2. Encuestas y polling:</strong> las proporciones muestrales son aproximadamente normales; los márgenes de error se calculan con el TLC. Las encuestas usan $n \approx 1000$–$1500$.</p>
<p style="margin:0 0 6px 0;"><strong>3. Finanzas:</strong> los retornos promedio de portafolios tienden a la normalidad; el Value at Risk (VaR) asume normalidad vía TLC. La diversificación reduce la variabilidad ($\sigma/\sqrt{n}$).</p>
<p style="margin:0 0 6px 0;"><strong>4. A/B Testing:</strong> diferencias de conversión entre grupos; base de las pruebas $z$ y $t$.</p>
<p style="margin:0;"><strong>5. Machine Learning:</strong> errores promedio en validación cruzada, intervalos de confianza para accuracy y comparación de modelos.</p>
</div>

## 5. Error Estándar

### 5.1 Definición y Conceptualización

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Error Estándar (Standard Error — SE)</p>
<p style="margin:0 0 6px 0;">El error estándar de un estadístico es la <strong>desviación estándar de su distribución muestral</strong>.</p>
<p style="margin:0 0 2px 0;"><strong>Para la media muestral:</strong></p>
$$SE(\bar{X}) = \frac{\sigma}{\sqrt{n}}$$
<p style="margin:6px 0 2px 0;">Cuando $\sigma$ es desconocido (caso típico), se estima con la desviación estándar muestral $s$:</p>
$$\widehat{SE}(\bar{X}) = \frac{s}{\sqrt{n}}$$
<p style="margin:6px 0 2px 0;"><strong>Para la proporción muestral:</strong></p>
$$SE(\hat{p}) = \sqrt{\frac{p(1-p)}{n}} \quad\text{o estimado:}\quad \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$
</div>

### 5.2 Interpretación Visual

La distribución muestral de $\bar{x}$ es aproximadamente normal y centrada en $\mu$. Las bandas de error estándar indican qué proporción de las medias muestrales caen cerca de $\mu$.

In [ ]:
# Distribución muestral de x_barra con bandas de error estándar
fig, ax = plt.subplots(figsize=(10, 4.5))
mu, se = 0, 1
x = np.linspace(-4, 4, 400)
y = stats.norm.pdf(x, mu, se)

ax.plot(x, y, color='#1565c0', lw=2)
ax.fill_between(x, y, where=(x >= -1) & (x <= 1),
                color='#ffcc80', alpha=0.6, label=r'$\pm 1\,SE$  (~68%)')
ax.fill_between(x, y, where=((x >= -1.96) & (x < -1)) | ((x > 1) & (x <= 1.96)),
                color='#ce93d8', alpha=0.5, label=r'$\pm 1.96\,SE$  (~95%)')
ax.axvline(mu, color='red', ls='--', lw=1.5)
ax.text(mu, 0.42, r'$\mu$', ha='center', color='red', fontsize=13)

ax.set_title(r'Distribución muestral de $\bar{x}$: bandas de error estándar', fontweight='bold')
ax.set_xlabel(r'$\bar{x}$  (en unidades de SE)')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.show()

**Reglas empíricas (distribución normal):**

- ≈ **68%** de las medias muestrales caen en $\mu \pm 1\,SE$.
- ≈ **95%** de las medias muestrales caen en $\mu \pm 1.96\,SE$.
- ≈ **99.7%** de las medias muestrales caen en $\mu \pm 3\,SE$.

### 5.3 Error Estándar vs Desviación Estándar

| Aspecto | Desviación Estándar ($s$) | Error Estándar (SE) |
|---|---|---|
| **¿Qué mide?** | Variabilidad de datos individuales | Variabilidad del estadístico $\bar{x}$ |
| **Fórmula** | $s = \sqrt{\dfrac{\sum(x_i - \bar{x})^2}{n-1}}$ | $SE = \dfrac{s}{\sqrt{n}}$ |
| **Efecto de $n$** | Converge a $\sigma$ (constante) | Disminuye: $\propto 1/\sqrt{n}$ |
| **Uso** | Describir dispersión de datos | Cuantificar precisión de estimación |
| **Contexto** | Descriptivo | Inferencial |
| **Magnitud** | Mayor | Menor (siempre $SE < s$) |

<div align="center"><em>Tabla 5.1 — Diferencias cruciales entre $s$ y $SE$.</em></div>

<div style="background-color:#fdecea; border-left:5px solid #c62828; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#b71c1c; font-size:1.05em;">⚠️&nbsp; Advertencia — Error Conceptual Común</p>
<p style="margin:0 0 6px 0;"><strong>No confundir:</strong></p>
<ul style="margin:0 0 8px 0;">
<li>$s$ = desviación estándar de LA MUESTRA (datos individuales).</li>
<li>$SE$ = error estándar de LA MEDIA (precisión del estimador).</li>
</ul>
<p style="margin:0;">$SE = s/\sqrt{n}$ es siempre menor que $s$. Aumentar $n$ reduce el $SE$ pero no necesariamente $s$: la desviación $s$ converge a $\sigma$; el error $SE$ converge a 0.</p>
</div>

## 6. Implementación en Python

### 6.1 Simulación: Concepto de Distribución Muestral

Construimos una población **uniforme** (no normal) y verificamos empíricamente que la distribución de $\bar{x}$ es normal (TLC) y que el error estándar empírico coincide con el teórico $\sigma/\sqrt{n}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)

print("="*70)
print("SIMULACIÓN: DISTRIBUCIÓN MUESTRAL DE LA MEDIA")
print("="*70)

# PASO 1: Crear población (uniforme - NO normal)
N = 100000
poblacion = np.random.uniform(low=0, high=100, size=N)
mu_pob = poblacion.mean()
sigma_pob = poblacion.std()

print(f"\nPOBLACIÓN (Distribución Uniforme):")
print(f"  Tamaño: {N:,}")
print(f"  Media (mu): {mu_pob:.2f}")
print(f"  Desv. Est. (sigma): {sigma_pob:.2f}")

# PASO 2: Tomar MUCHAS muestras y calcular medias
n = 30
num_muestras = 10000

medias_muestrales = []
for i in range(num_muestras):
    muestra = np.random.choice(poblacion, size=n, replace=True)
    medias_muestrales.append(muestra.mean())

medias_muestrales = np.array(medias_muestrales)

# PASO 3: Analizar distribución muestral
media_de_medias = medias_muestrales.mean()
se_empirico = medias_muestrales.std()
se_teorico = sigma_pob / np.sqrt(n)

print(f"\nDISTRIBUCIÓN MUESTRAL de x_barra (n={n}):")
print(f"  E[x_barra] = {media_de_medias:.2f} (teórico: mu={mu_pob:.2f})")
print(f"  SE empírico = {se_empirico:.2f}")
print(f"  SE teórico = sigma/sqrt(n) = {se_teorico:.2f}")

# Test de normalidad
stat, p_value = stats.shapiro(medias_muestrales[:5000])
print(f"\nTest Shapiro-Wilk:")
print(f"  p-valor: {p_value:.6f}")
if p_value > 0.05:
    print(f"  Distribución aproximadamente NORMAL (TLC validado)")
else:
    print(f"  No completamente normal")

print("\nCONCLUSIÓN:")
print("Aunque la población es UNIFORME (no normal),")
print("la distribución de x_barra es NORMAL (TLC)!")

**Visualización de la simulación** (reutiliza `poblacion` y `medias_muestrales`):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Población original (uniforme, plana)
axes[0].hist(poblacion, bins=50, color='#90caf9', edgecolor='white')
axes[0].set_title('Población (Uniforme)', fontweight='bold')
axes[0].set_xlabel('Valor'); axes[0].set_ylabel('Frecuencia')

# Distribución muestral de x_barra (normal por TLC)
axes[1].hist(medias_muestrales, bins=40, density=True, color='#a5d6a7', edgecolor='white')
x = np.linspace(medias_muestrales.min(), medias_muestrales.max(), 200)
axes[1].plot(x, stats.norm.pdf(x, medias_muestrales.mean(), medias_muestrales.std()),
             'r-', lw=2, label='Normal teórica')
axes[1].axvline(mu_pob, color='red', ls='--', label=fr'$\mu$ = {mu_pob:.1f}')
axes[1].set_title(r'Distribución muestral de $\bar{x}$ (n=30)', fontweight='bold')
axes[1].set_xlabel(r'$\bar{x}$'); axes[1].set_ylabel('Densidad'); axes[1].legend()

plt.tight_layout()
plt.show()

### 6.2 Demostración del TLC: Diferentes Tamaños de Muestra

Cuantificamos la convergencia midiendo la **asimetría** de la distribución muestral a medida que aumenta $n$ (complementa la figura de la Sección 4.2).

In [ ]:
np.random.seed(123)

# Población exponencial (altamente asimétrica)
poblacion_exp = np.random.exponential(scale=10, size=100000)
mu = poblacion_exp.mean()
sigma = poblacion_exp.std()

print("DEMOSTRACIÓN DEL TEOREMA DEL LÍMITE CENTRAL")
print(f"Población Exponencial:")
print(f"  mu = {mu:.2f}, sigma = {sigma:.2f}")
print(f"  Asimetría = {stats.skew(poblacion_exp):.2f}")

tamanos = [2, 5, 10, 30, 100]
num_muestras = 10000

print("\nCONVERGENCIA A NORMALIDAD:")
for n in tamanos:
    medias = []
    for _ in range(num_muestras):
        muestra = np.random.choice(poblacion_exp, size=n, replace=True)
        medias.append(muestra.mean())
    medias = np.array(medias)
    asim = stats.skew(medias)
    estado = 'Casi normal' if abs(asim) < 0.2 else 'Aún asimétrica'
    print(f"  n={n:3d}: Asimetría de x_barra = {asim:6.3f}  {estado}")

### 6.3 Cálculo de Intervalos de Confianza

A partir de UNA muestra observada, construimos el intervalo de confianza del 95% para $\mu$ usando el TLC, y comparamos distintos niveles de confianza.

In [ ]:
print("INTERVALOS DE CONFIANZA BASADOS EN EL TLC")

np.random.seed(999)
n_muestra = 100
muestra_real = np.random.choice(poblacion_exp, size=n_muestra, replace=True)

x_barra = muestra_real.mean()
s = muestra_real.std(ddof=1)
se_estimado = s / np.sqrt(n_muestra)

print(f"\nMUESTRA OBSERVADA (n={n_muestra}):")
print(f"  Media muestral (x_barra): {x_barra:.2f}")
print(f"  Desv. est. muestral (s): {s:.2f}")
print(f"  Error estándar estimado (SE): {se_estimado:.2f}")

# IC 95% usando distribución normal (TLC, n grande)
z_critico = stats.norm.ppf(0.975)  # 1.96
margen_error_95 = z_critico * se_estimado
ic_inf_95 = x_barra - margen_error_95
ic_sup_95 = x_barra + margen_error_95

print(f"\nINTERVALO DE CONFIANZA 95%:")
print(f"  IC = [{ic_inf_95:.2f}, {ic_sup_95:.2f}]")
print(f"  Margen de error: +/-{margen_error_95:.2f}")

if ic_inf_95 <= mu <= ic_sup_95:
    print(f"  El IC CONTIENE mu = {mu:.2f} (éxito!)")
else:
    print(f"  El IC NO contiene mu (parte del 5% esperado)")

# Comparar niveles de confianza
print("\nCOMPARACIÓN DE NIVELES DE CONFIANZA:")
for conf in [0.90, 0.95, 0.99]:
    z = stats.norm.ppf((1 + conf) / 2)
    me = z * se_estimado
    print(f"  {conf*100:.0f}%: z={z:.3f}, ME=+/-{me:.2f}, " +
          f"IC=[{x_barra-me:.2f}, {x_barra+me:.2f}]")

print("\nTrade-off: Mayor confianza -> Intervalo más ancho")

**Interpretación correcta del nivel de confianza.** Un IC del 95% **no** significa que haya 95% de probabilidad de que $\mu$ esté en ese intervalo particular. Significa que, si repitiéramos el muestreo muchas veces, el **95% de los intervalos** construidos contendría a $\mu$. La siguiente figura lo demuestra: construimos 50 intervalos y contamos cuántos atrapan el verdadero $\mu$.

In [ ]:
# Cobertura de intervalos de confianza: 50 muestras independientes
np.random.seed(7)
mu_verdadero = poblacion_exp.mean()
n = 100
k = 50
z = stats.norm.ppf(0.975)

fig, ax = plt.subplots(figsize=(11, 6))
cubiertos = 0
for i in range(k):
    m = np.random.choice(poblacion_exp, size=n, replace=True)
    xb = m.mean()
    se = m.std(ddof=1) / np.sqrt(n)
    lo, hi = xb - z*se, xb + z*se
    contiene = lo <= mu_verdadero <= hi
    cubiertos += contiene
    color = '#2e7d32' if contiene else '#c62828'
    ax.plot([lo, hi], [i, i], color=color, lw=1.6)
    ax.plot(xb, i, 'o', color=color, ms=3)

ax.axvline(mu_verdadero, color='black', ls='--', lw=1.5,
           label=fr'$\mu$ verdadero = {mu_verdadero:.2f}')
ax.set_title(f'50 intervalos de confianza 95%: {cubiertos} contienen $\\mu$  ({cubiertos*100//k}%)',
             fontweight='bold')
ax.set_xlabel('Valor'); ax.set_ylabel('Muestra #')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 7. Aplicaciones Prácticas

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Control de Calidad en Manufactura</p>
<p style="margin:0 0 6px 0;"><strong>Problema:</strong> estimar el tiempo promedio de producción por unidad.</p>
<p style="margin:0 0 6px 0;"><strong>Datos</strong> (muestra $n=50$): $\bar{x} = 12.3$ min, $s = 2.1$ min, $SE = 2.1/\sqrt{50} = 0.297$ min.</p>
<p style="margin:0 0 6px 0;"><strong>IC 95%:</strong> $[12.3 - 1.96(0.297),\; 12.3 + 1.96(0.297)] = [11.72,\; 12.88]$ min.</p>
<p style="margin:0 0 6px 0;"><strong>Decisión:</strong> con 95% de confianza, el tiempo promedio está entre 11.7 y 12.9 min.</p>
<p style="margin:0;"><strong>Acción:</strong> planificar capacidad sobre la base de 12.5 min/unidad (punto medio + margen).</p>
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Encuestas de Opinión</p>
<p style="margin:0 0 6px 0;"><strong>Problema:</strong> estimar la intención de voto para el candidato A.</p>
<p style="margin:0 0 6px 0;"><strong>Datos:</strong> $n = 1500$ votantes; 780 apoyan al candidato A $\Rightarrow \hat{p} = 0.52$.</p>
<p style="margin:0 0 6px 0;"><strong>Error estándar:</strong> $SE(\hat{p}) = \sqrt{\dfrac{0.52 \times 0.48}{1500}} = 0.0129$.</p>
<p style="margin:0 0 6px 0;"><strong>IC 95%:</strong> $[0.495,\; 0.545]$.</p>
<p style="margin:0;"><strong>Reporte:</strong> «52% ± 2.5%» (margen de error típico reportado).</p>
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — A/B Testing en Tecnología</p>
<p style="margin:0 0 6px 0;"><strong>Problema:</strong> ¿el nuevo diseño aumenta la tasa de conversión?</p>
<p style="margin:0 0 6px 0;"><strong>Diseño A:</strong> $n_A = 5000$, 350 conversiones, $\hat{p}_A = 7.0\%$, $SE_A = 0.0036$.</p>
<p style="margin:0 0 6px 0;"><strong>Diseño B:</strong> $n_B = 5000$, 425 conversiones, $\hat{p}_B = 8.5\%$, $SE_B = 0.0039$.</p>
<p style="margin:0 0 6px 0;"><strong>SE de la diferencia:</strong> $SE(\text{dif}) = \sqrt{0.0036^2 + 0.0039^2} = 0.0053$.</p>
<p style="margin:0 0 6px 0;"><strong>IC 95% para la diferencia:</strong> $[0.0046,\; 0.0254]$.</p>
<p style="margin:0;"><strong>Decisión:</strong> el IC no incluye 0 → diferencia estadísticamente significativa. Implementar el diseño B.</p>
</div>

## 8. Planificación del Tamaño Muestral

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Fórmula para Determinar el $n$ Necesario</p>
<p style="margin:0 0 6px 0;">Para estimar $\mu$ con margen de error $ME$ y nivel de confianza $(1-\alpha)$:</p>
$$n = \left(\frac{z_{\alpha/2}\,\sigma}{ME}\right)^2$$
<p style="margin:6px 0 0 0;">Donde $z_{\alpha/2}$ es el valor crítico (1.96 para 95%, 2.576 para 99%), $\sigma$ es la desviación estándar poblacional (estimada de un estudio piloto) y $ME$ es el margen de error deseado. Siempre redondear $n$ hacia arriba.</p>
</div>

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Planificación Muestral</p>
<p style="margin:0 0 6px 0;">Una empresa quiere estimar el tiempo promedio de respuesta con $\pm 30$ segundos de margen, confianza 95% y $\sigma \approx 120$ segundos.</p>
$$n = \left(\frac{1.96 \times 120}{30}\right)^2 = (7.84)^2 = 61.47 \approx 62$$
<p style="margin:6px 0 0 0;"><strong>Respuesta:</strong> necesitan $n \geq 62$ observaciones.</p>
</div>

In [ ]:
def calcular_n_necesario(margen_error, sigma, confianza=0.95):
    """
    Calcula el tamaño muestral necesario para estimar una media.

    Parámetros
    ----------
    margen_error : float  — Margen de error deseado
    sigma        : float  — Desviación estándar poblacional (estimada)
    confianza    : float  — Nivel de confianza (default: 0.95)

    Retorna
    -------
    n : int  — Tamaño de muestra necesario
    """
    from scipy import stats
    import numpy as np

    alpha = 1 - confianza
    z = stats.norm.ppf(1 - alpha/2)
    n = (z * sigma / margen_error) ** 2
    return int(np.ceil(n))

# Ejemplo de uso
n_requerido = calcular_n_necesario(margen_error=30, sigma=120, confianza=0.95)
print(f"Tamaño muestral necesario: {n_requerido}")

# Análisis de sensibilidad
print("\nANÁLISIS DE SENSIBILIDAD (sigma=120, confianza=95%):")
for me in [20, 30, 40, 50]:
    n = calcular_n_necesario(me, 120, 0.95)
    print(f"  ME = +/-{me:2d}s -> n = {n:3d}")

print("\nPrecisión tiene costo cuadrático.")

## 9. Mejores Prácticas y Errores Comunes

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Guía de Mejores Prácticas</p>
<p style="margin:0 0 6px 0;"><strong>1. Verificar supuestos del TLC:</strong> muestreo aleatorio simple, observaciones independientes, $n \geq 30$ para poblaciones asimétricas. Si $n < 30$, usar la distribución $t$ de Student.</p>
<p style="margin:0 0 6px 0;"><strong>2. Reportar medidas de incertidumbre:</strong> no basta con decir «la media es 25». Lo correcto es «la media es 25 (SE = 1.5)». Lo óptimo es «la media es 25, IC 95%: [22, 28]».</p>
<p style="margin:0 0 6px 0;"><strong>3. Interpretar los IC correctamente:</strong> un IC del 95% NO significa «95% de probabilidad de que $\mu$ esté en [a, b]». Significa que el 95% de los IC construidos con este método contendrán $\mu$. El parámetro es fijo; el intervalo es aleatorio.</p>
<p style="margin:0 0 6px 0;"><strong>4. Considerar significancia práctica:</strong> muestras grandes hacen que diferencias triviales sean estadísticamente significativas. Reportar el tamaño del efecto además del p-valor.</p>
<p style="margin:0;"><strong>5. Reproducibilidad:</strong> usar <code>np.random.seed()</code> en simulaciones. Documentar el código y las decisiones metodológicas.</p>
</div>

<div style="background-color:#fdecea; border-left:5px solid #c62828; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#b71c1c; font-size:1.05em;">⚠️&nbsp; Advertencia — Errores Comunes a Evitar</p>
<ol style="margin:0;">
<li><strong>Confundir $s$ y $SE$:</strong> $s$ = variabilidad de los datos; $SE$ = precisión de la estimación. Siempre $SE < s$.</li>
<li><strong>Aplicar el TLC con $n$ pequeño:</strong> si $n < 30$ Y la población es asimétrica, el TLC no garantiza normalidad. Usar la distribución $t$ o métodos no paramétricos.</li>
<li><strong>Ignorar la independencia:</strong> observaciones correlacionadas violan los supuestos (ej. medir el mismo sujeto varias veces).</li>
<li><strong>Muestreo no aleatorio:</strong> el sesgo de selección invalida toda la inferencia.</li>
<li><strong>Sobre-interpretación:</strong> el IC no dice dónde está $\mu$ específicamente; solo cuantifica la incertidumbre del método.</li>
</ol>
</div>

## 10. Tabla de Referencia Rápida

| Concepto | Fórmula / Valor | Uso |
|---|---|---|
| Media muestral | $\bar{x} = \frac{1}{n}\sum x_i$ | Estimador puntual de $\mu$ |
| Error estándar | $SE = \frac{s}{\sqrt{n}}$ | Precisión del estimador |
| IC 95% ($n$ grande) | $\bar{x} \pm 1.96 \times SE$ | Rango plausible para $\mu$ |
| IC 99% ($n$ grande) | $\bar{x} \pm 2.576 \times SE$ | Mayor confianza, más ancho |
| Margen de error | $ME = z_{\alpha/2} \times SE$ | Mitad del ancho del IC |
| TLC (regla) | $n \geq 30$ | Suficiente para normalidad |
| Tamaño muestral | $n = \left(\frac{z\,\sigma}{ME}\right)^2$ | Planificación de estudio |
| SE proporción | $\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$ | Para variables binarias |
| $z$ para 90% | 1.645 | Valor crítico |
| $z$ para 95% | 1.96 | Valor crítico (más usado) |
| $z$ para 99% | 2.576 | Valor crítico |

<div align="center"><em>Tabla 10.1 — Fórmulas y valores clave (referencia rápida).</em></div>

## 11. Conclusiones

La inferencia estadística, sustentada por el Teorema del Límite Central, es el puente que conecta observaciones limitadas (muestras) con conocimiento generalizable (población). Es la esencia del método científico cuantitativo y la base de la toma de decisiones basada en datos.

**Mensajes clave para recordar:**

1. **Población vs Muestra:** parámetros (desconocidos) vs estadísticos (observados).
2. **Distribuciones muestrales:** cuantifican la variabilidad de los estadísticos de muestra en muestra.
3. **TLC — el gran unificador:** garantiza la normalidad de $\bar{X}$ para $n$ grande, sin importar la forma poblacional.
4. **Error estándar:** $SE = \sigma/\sqrt{n}$ mide la precisión y disminuye con $\sqrt{n}$.
5. **Intervalos de confianza:** cuantifican la incertidumbre de manera formal y comunicable.
6. **Ley de rendimientos decrecientes:** para reducir el SE a la mitad, se necesitan 4× más datos.
7. **Muestreo aleatorio:** no negociable para una inferencia válida.
8. **Simulación:** herramienta poderosa para validar la teoría y explorar el comportamiento.

Estos conceptos son la base de toda la estadística inferencial moderna: pruebas de hipótesis, regresión, ANOVA, diseño experimental y machine learning. La maestría en inferencia permite no solo analizar datos, sino cuantificar la incertidumbre, comunicar hallazgos con rigor y tomar decisiones informadas bajo condiciones de información imperfecta.

### Cierre

<div style="background-color:#ffffff; border:1px solid #d9d9d9; border-left:60px solid #0d2741; border-radius:6px; padding:16px 20px; margin:6px 0;">
<span style="font-size:1.4em;">💡</span>&nbsp; El Teorema del Límite Central no es solo un resultado matemático; es la razón por la que podemos hacer ciencia cuantitativa con muestras limitadas. Es lo que permite que las encuestas políticas predigan elecciones, que los estudios médicos salven vidas y que las empresas optimicen sus operaciones. Dominarlo es fundamental para cualquier científico de datos. Con las herramientas y conceptos de este apunte, estás equipado para cuantificar la incertidumbre, comunicar hallazgos con rigor y tomar decisiones informadas.
</div>

---

## Referencias bibliográficas

- Casella, G., & Berger, R. L. (2002). *Statistical inference* (2nd ed.). Duxbury Press.
- DeGroot, M. H., & Schervish, M. J. (2012). *Probability and statistics* (4th ed.). Pearson.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning* (2nd ed.). Springer.
- Rice, J. A. (2007). *Mathematical statistics and data analysis* (3rd ed.). Duxbury Press.
- Wasserman, L. (2004). *All of statistics: A concise course in statistical inference*. Springer.